<a href="https://colab.research.google.com/github/toxzak-svg/ingexuity/blob/main/finetune_mamba_370m.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IngExuity Fine-Tune: Mamba-370M on Conversational Data

## Goal
Fine-tune Mamba-370M on Zach's conversation data to give IngExuity
personal voice and fluency. Deploy to Railway after.

## Hardware
- Colab T4 (16GB VRAM) — fine-tune here
- QLoRA / INT8 quantization to fit in VRAM
- Target: ~50M tokens, 3 epochs, overnight

## Output
- Saved to `toxzak/ingexuity-370m` on HuggingFace
- Ready for INT4 quantization + Railway deploy

In [ ]:
# @title Cell 1: Setup
!pip install -q bitsandbytes transformers accelerate datasets hf-transfer
!pip install -q sentencepiece huggingface_hub

import os
os.environ["HF_TOKEN"] = ""  # Set your token or use Secrets
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("Done")

In [ ]:
# @title Cell 2: Load Model + Tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "state-spaces/mamba-370m"

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")

In [ ]:
# @title Cell 3: Prepare Dataset
# Replace with your conversation JSONL data
# Format: one JSON per line: {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}

from datasets import load_dataset

# Option A: Load from HuggingFace (Zach's dataset)
dataset = load_dataset("ZacharyMaronek/starfire-personal-v1", split="train")

def format_conversation(example):
    """Format as a single string with user/assistant turns."""
    messages = example.get("messages", [])
    text = ""
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        if role == "user":
            text += f"User: {content}\n"
        elif role == "assistant":
            text += f"Assistant: {content}\n"
    return {"text": text.strip()}

dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)

# Tokenize
def tokenize(example):
    enc = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

dataset = dataset.map(tokenize, batched=False, remove_columns=["text"])
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")
print(f"Sample: {dataset['train'][0]}")

In [ ]:
# @title Cell 4: Training Config
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_int8_training

model = prepare_model_for_int8_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["x_proj", "dt_proj", "out_proj"],  # Mamba SSM modules
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked
)

training_args = TrainingArguments(
    output_dir="./mamba-370m-voice",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch = 16
    optim="adamw_torch",
    learning_rate=1e-4,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_checkpoint": True},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
)

In [ ]:
# @title Cell 5: Train
trainer.train()
print("Training done!")

In [ ]:
# @title Cell 6: Merge + Save
# Merge LoRA adapters into base model
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./ingexuity-370m-final")
tokenizer.save_pretrained("./ingexuity-370m-final")
print("Saved to ./ingexuity-370m-final")

# Push to HuggingFace (requires HF_TOKEN in Secrets or Cell 1)
merged_model.push_to_hub("toxzak/ingexuity-370m", token=os.environ.get("HF_TOKEN"))
tokenizer.push_to_hub("toxzak/ingexuity-370m", token=os.environ.get("HF_TOKEN"))
print("Pushed to HuggingFace: toxzak/ingexuity-370m")

In [ ]:
# @title Cell 7: Test Inference
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="./ingexuity-370m-final",
    tokenizer=tokenizer,
    device=-1,  # CPU
    max_new_tokens=128,
    temperature=0.7,
)

prompt = "User: Hey, how are you?\nAssistant:"
output = pipe(prompt, do_sample=True, top_p=0.9)
print(output[0]["generated_text"])